In [ ]:
# === Setup ===
import os, random
import numpy as np
import torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    print('Running on Kaggle...')
else:
    print('Running locally...')


# Hướng dẫn giải (Reference Solution)
ĐÂY LÀ MỘT PIPELINE MẪU CHUẨN MỰC BẠN NÊN LƯU LẠI.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

# MÔ PHỎNG: Dataset Dummy để code có thể chạy ngay
class DummyDataset(Dataset):
    def __init__(self, num_samples=100, transform=None):
        self.num_samples = num_samples
        self.transform = transform
    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        # Tạo ảnh nhiễu 3 kênh 224x224 mô phỏng
        img = torch.randn(3, 224, 224)
        label = torch.randint(0, 2, (1,)).item()
        return img, label

# 1. Transform chuẩn ImageNet
transform = transforms.Compose([
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(DummyDataset(num_samples=16), batch_size=4, shuffle=True)

# 2. Khởi tạo Model
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(512, 2)

# 3. Train Loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

model.train()
for epoch in range(2):
    total_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1} - Loss: {total_loss/len(train_loader):.4f}')
